---
# Section 1 — Setup

In [ ]:
# ── Install dependencies ──────────────────────────────────────────
!pip install -q timm
import os, sys, json

os.makedirs('/root/.kaggle', exist_ok=True)
token = {"username": "YOUR_KAGGLE_USERNAME", "key": "YOUR_KAGGLE_KEY"}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(token, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d . && echo "Unzipped successfully"

# ── Imports ───────────────────────────────────────────────────────
import time, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.utils import make_grid

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    roc_curve, auc, classification_report
)

import timm

# ── Global config ─────────────────────────────────────────────────
DATA_ROOT    = './chest_xray'
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASSES      = ['NORMAL', 'PNEUMONIA']
IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 42

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if DEVICE == 'cuda': torch.cuda.manual_seed_all(SEED)
print(f'PyTorch {torch.__version__} | timm {timm.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')

# ── Verify dataset paths ──────────────────────────────────────────
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        path = os.path.join(DATA_ROOT, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 'PATH NOT FOUND'
        print(f'  {split}/{cls}: {count}')

# ── Image preprocessing & transforms ─────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── DataLoaders ───────────────────────────────────────────────────
train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, 'train'), train_transforms)
val_ds   = datasets.ImageFolder(os.path.join(DATA_ROOT, 'val'),   val_transforms)
test_ds  = datasets.ImageFolder(os.path.join(DATA_ROOT, 'test'),  val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Class map: {train_ds.class_to_idx}')

# ── Shared training utilities ─────────────────────────────────────
def get_class_weights():
    counts  = np.array([len(os.listdir(os.path.join(DATA_ROOT, 'train', c))) for c in CLASSES])
    weights = counts.sum() / (len(counts) * counts)
    weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f'Class weights → NORMAL: {weights[0]:.3f}, PNEUMONIA: {weights[1]:.3f}')
    return weights

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        probs  = torch.softmax(logits, dim=1)[:, 1]
        total_loss += loss.item() * imgs.size(0)
        preds  = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels, all_probs

def run_training(model, model_name, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    weights   = get_class_weights()
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        print(f'[{model_name}] Ep {epoch:02d}/{epochs}  '
              f'TrLoss={tr_loss:.4f} TrAcc={tr_acc:.4f}  '
              f'VlLoss={vl_loss:.4f} VlAcc={vl_acc:.4f}  ({time.time()-t0:.1f}s)')

    model.load_state_dict(best_state)
    print(f'Best val acc: {best_val_acc:.4f}')
    return model, history

# ── Sample image grid ─────────────────────────────────────────────
imgs, labels = next(iter(train_loader))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
imgs_denorm = (imgs[:8] * std + mean).clamp(0, 1)
grid = make_grid(imgs_denorm, nrow=4)
plt.figure(figsize=(12, 4))
plt.imshow(grid.permute(1, 2, 0))
plt.title('Sample training images: ' + ' | '.join([CLASSES[l] for l in labels[:8].tolist()]))
plt.axis('off')
plt.show()
print('Section 1 complete.')

---
# Section 2 — Base Model: DenseNet-121

In [ ]:
# ── Build DenseNet-121 ────────────────────────────────────────────
def build_densenet121(num_classes=2):
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)
    )
    # Unfreeze last dense block + norm + head
    for param in model.features.denseblock4.parameters(): param.requires_grad = True
    for param in model.features.norm5.parameters():       param.requires_grad = True
    for param in model.classifier.parameters():           param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'DenseNet-121 | Trainable: {trainable:,} / {total:,} params')
    return model.to(DEVICE)

# ── Train ─────────────────────────────────────────────────────────
base_model = build_densenet121()
base_model, base_history = run_training(
    base_model, 'DenseNet121', train_loader, val_loader, epochs=EPOCHS
)

# ── Evaluate ──────────────────────────────────────────
criterion_eval = nn.CrossEntropyLoss()
_, _, base_preds, base_labels, base_probs = evaluate(base_model, test_loader, criterion_eval)

base_metrics = {
    'accuracy' : accuracy_score(base_labels, base_preds),
    'precision': precision_score(base_labels, base_preds, pos_label=1),
    'recall'   : recall_score(base_labels, base_preds, pos_label=1),
    'f1'       : f1_score(base_labels, base_preds, pos_label=1),
    'auc_roc'  : roc_auc_score(base_labels, base_probs),
}
print('\n── DenseNet-121 Test Results ──')
for k, v in base_metrics.items(): print(f'  {k:<12}: {v:.4f}')
print()
print(classification_report(base_labels, base_preds, target_names=CLASSES))
print('Section 2 complete.')

---
# Section 3 — Advanced Model: DINOv2-Base (2024)

In [ ]:
# ── Build DINOv2-Base ─────────────────────────────────────────────
def build_dinov2(num_classes=2):
    backbone = timm.create_model(
        'vit_base_patch14_dinov2.lvd142m',
        pretrained=True,
        num_classes=0,
        img_size=IMG_SIZE,
    )

    # Freeze backbone
    for param in backbone.parameters():
        param.requires_grad = False

    embed_dim = backbone.embed_dim

    # Wrap backbone + head
    class DINOv2Classifier(nn.Module):
        def __init__(self, backbone, embed_dim, num_classes):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, 256),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes)
            )

        def forward(self, x):
            features = self.backbone(x)
            return self.head(features)

    model = DINOv2Classifier(backbone, embed_dim, num_classes)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'DINOv2-Base | Trainable (head only): {trainable:,} / {total:,} params')
    return model.to(DEVICE)

adv_model = build_dinov2(num_classes=2)

# ── Stage 1: Head only (3 epochs, frozen backbone) ─────────
print('\n── Stage 1: Head warm-up (backbone frozen) ──')
adv_model, _ = run_training(
    adv_model, 'DINOv2_warmup', train_loader, val_loader,
    epochs=3, lr=1e-3
)

# ── Stage 2: Full model ────────────────────────────────────
print('\n── Stage 2: Full fine-tune (all layers, low LR) ──')
for param in adv_model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in adv_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in adv_model.parameters())
print(f'DINOv2-Base | Trainable (full): {trainable:,} / {total:,} params')

adv_model, adv_history = run_training(
    adv_model, 'DINOv2_Base', train_loader, val_loader,
    epochs=EPOCHS, lr=5e-5
)

# ── Evaluate ──────────────────────────────────────────────
_, _, adv_preds, adv_labels, adv_probs = evaluate(adv_model, test_loader, nn.CrossEntropyLoss())

adv_metrics = {
    'accuracy' : accuracy_score(adv_labels, adv_preds),
    'precision': precision_score(adv_labels, adv_preds, pos_label=1),
    'recall'   : recall_score(adv_labels, adv_preds, pos_label=1),
    'f1'       : f1_score(adv_labels, adv_preds, pos_label=1),
    'auc_roc'  : roc_auc_score(adv_labels, adv_probs),
}
print('\n── DINOv2-Base Test Results ──')
for k, v in adv_metrics.items(): print(f'  {k:<12}: {v:.4f}')
print()
print(classification_report(adv_labels, adv_preds, target_names=CLASSES))
print('Section 3 complete.')

---
# Section 4 — Results

In [ ]:
metric_keys   = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']

# ── Fig 1: Curves ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Training Curves', fontsize=15, fontweight='bold')

for col, (name, hist, color) in enumerate([
    ('DenseNet-121', base_history, '#2563EB'),
    ('DINOv2-Base',  adv_history,  '#DC2626'),
]):
    epochs_range = range(1, len(hist['train_loss']) + 1)
    axes[0][col].plot(epochs_range, hist['train_loss'], color=color, linewidth=2, label='Train')
    axes[0][col].plot(epochs_range, hist['val_loss'],   color=color, linewidth=2, linestyle='--', alpha=0.7, label='Val')
    axes[0][col].set_title(f'{name} — Loss'); axes[0][col].set_xlabel('Epoch'); axes[0][col].set_ylabel('Loss')
    axes[0][col].legend(); axes[0][col].grid(True, alpha=0.3)

    axes[1][col].plot(epochs_range, hist['train_acc'], color=color, linewidth=2, label='Train')
    axes[1][col].plot(epochs_range, hist['val_acc'],   color=color, linewidth=2, linestyle='--', alpha=0.7, label='Val')
    axes[1][col].set_title(f'{name} — Accuracy'); axes[1][col].set_xlabel('Epoch'); axes[1][col].set_ylabel('Accuracy')
    axes[1][col].set_ylim(0, 1.05); axes[1][col].legend(); axes[1][col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Fig 2: Confusion Matrices ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Confusion Matrices (Test Set)', fontsize=13, fontweight='bold')

for ax, (name, preds, labels) in zip(axes, [
    ('DenseNet-121', base_preds, base_labels),
    ('DINOv2-Base',  adv_preds,  adv_labels),
]):
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, annot_kws={'size': 14})
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(name); ax.set_xlabel(f'Predicted\nTN={tn}  FP={fp}  FN={fn}  TP={tp}')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

# ── Fig 3: ROC Curves ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
for name, labels, probs, color in [
    ('DenseNet-121', base_labels, base_probs, '#2563EB'),
    ('DINOv2-Base',  adv_labels,  adv_probs,  '#DC2626'),
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc(fpr,tpr):.4f})')

ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set'); ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Fig 4: Metric Comparison Bar Chart ───────────────────────────
x = np.arange(len(metric_keys)); width = 0.32

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, [base_metrics[k] for k in metric_keys], width, label='DenseNet-121', color='#2563EB', alpha=0.88)
bars2 = ax.bar(x + width/2, [adv_metrics[k]  for k in metric_keys], width, label='DINOv2-Base',  color='#DC2626', alpha=0.88)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Set'); ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Summary Table ─────────────────────────────────────────────────
print('='*62)
print(f'{"Metric":<14} {"DenseNet-121":>14} {"DINOv2-Base":>14} {"Δ":>8}')
print('-'*62)
for k, label in zip(metric_keys, metric_labels):
    b, a = base_metrics[k], adv_metrics[k]
    arrow = '▲' if a > b else '▼'
    print(f'{label:<14} {b:>14.4f} {a:>14.4f} {arrow}{abs(a-b):>6.4f}')
print('='*62)
print('Section 4 complete.')